# Trace-length analysis

Compare completion length distributions across three checkpoints (base, sft, grpo) on GSM8K test. This is what the length-penalty story in `docs/over_reasoning_failure_modes.md` looks like.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

runs = Path('../benchmarks/runs')

def lens(tag):
    p = runs / f'gsm8k.{tag}.json'
    if not p.exists():
        return np.array([])
    rows = json.loads(p.read_text()).get('rows', [])
    return np.array([len(r.get('trace','')) // 4 for r in rows], dtype=float)

base = lens('base'); sft = lens('sft'); grpo = lens('grpo')
df = pd.DataFrame({'stage': ['base']*len(base)+['sft']*len(sft)+['grpo']*len(grpo),
                   'approx_tokens': np.concatenate([base, sft, grpo])})
df.groupby('stage')['approx_tokens'].describe()

In [ ]:
for stage, arr in [('base', base), ('sft', sft), ('grpo', grpo)]:
    if len(arr):
        plt.hist(arr, bins=40, alpha=0.4, label=stage)
plt.axvline(512, ls='--')
plt.axvline(1024, ls='--')
plt.xlabel('approx tokens per completion')
plt.ylabel('count')
plt.legend(); plt.title('gsm8k completion length by stage'); plt.show()